In [ ]:
# ============================================================
# Cell 0 - Install fixed compatible versions
# Run once, then RESTART Kaggle session/runtime
# ============================================================

!pip uninstall -y torch torchvision torchaudio

!pip install --no-cache-dir \
    torch==2.5.1+cu121 \
    torchvision==0.20.1+cu121 \
    torchaudio==2.5.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
    transformers==4.51.3 \
    datasets==3.5.0 \
    accelerate==1.6.0 \
    peft==0.15.2 \
    trl==0.17.0 \
    bitsandbytes==0.45.5 \
    sentencepiece==0.2.0

print("Install finished. Now restart the Kaggle session/runtime, then continue from Cell 1.")

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 187.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 151.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 110.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 249.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 174.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# ============================================================
# Cell 1 - Check environment
# ============================================================

import torch
import transformers
import datasets
import accelerate
import peft
import trl
import bitsandbytes as bnb

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bnb.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Torch: 2.5.1+cu121
Transformers: 4.51.3
Datasets: 3.5.0
Accelerate: 1.6.0
PEFT: 0.15.2
TRL: 0.17.0
BitsAndBytes: 0.45.5
CUDA available: True
GPU: Tesla T4
GPU capability: (7, 5)
GPU memory GB: 14.56


In [ ]:
# ============================================================
# Cell 2 - Load and format dataset
# ============================================================

from datasets import load_dataset

dataset = load_dataset("ruslanmv/HealthCareMagic-100k", split="train")

print(dataset)
print(dataset.column_names)
print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/478 [00:00<?, ?B/s]

healthcaremagic.parquet:   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 112165
})
['instruction', 'input', 'output']
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In thi

In [ ]:
# ============================================================
# Cell 3 - Convert dataset to single text column + choose subset
# ============================================================

SEED = 42

TRAIN_SIZE = 5_000
EVAL_SIZE = 500

def format_example(example):
    instruction = str(example.get("instruction", "")).strip()
    input_text = str(example.get("input", "")).strip()
    output_text = str(example.get("output", "")).strip()

    if input_text:
        text = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output_text}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{output_text}"""

    return {"text": text}

dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
    desc="Formatting dataset"
)

dataset = dataset.shuffle(seed=SEED)

train_dataset = dataset.select(range(TRAIN_SIZE))
eval_dataset = dataset.select(range(TRAIN_SIZE, TRAIN_SIZE + EVAL_SIZE))

print("Full formatted dataset:", len(dataset))
print("Train dataset:", len(train_dataset))
print("Eval dataset:", len(eval_dataset))

print(train_dataset[0]["text"][:1200])

Formatting dataset:   0%|          | 0/112165 [00:00<?, ? examples/s]

Full formatted dataset: 112165
Train dataset: 5000
Eval dataset: 500
### Instruction:
If you are a doctor, please answer the medical questions based on the patient's description.

### Input:
I have been having alot of catching ,pain and discomfort under my right rib.  If I twist to either side especially my right it feels like my rib actually catches on something and at times I have to stop try to catch my breath and wait for it to subside.  There are times if I am laughing too hard that it will do the same thing but normally its more so if I have twisted or moved  a certain way

### Response:
Hi thanks for asking question. Here you are complaining pain in particular position esp. While turning to a side. So strong possibility is about moderate degree muscular strain. It might have occurred by heavyweight lift or during some activities. Simple analgesic taken. Take rest. Sleep in supine position. Second here Costco Chat Doctor.  Ribs are tender to touch.x-ray also useful. If cough, col

In [ ]:
# ============================================================
# Cell 4 - Load tokenizer and model in 4-bit
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

print("Model loaded successfully.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
# ============================================================
# Cell 5 - LoRA configuration
# ============================================================

from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",

    # Phi-3 uses fused projection names
    target_modules=["qkv_proj", "o_proj"]
)

print(lora_config)

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'qkv_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)


In [ ]:
# ============================================================
# Cell 6 - SFT configuration
# ============================================================

from trl import SFTConfig

training_config = SFTConfig(
    output_dir="/kaggle/working/medical-model",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,

    # This will be ignored because max_steps is set
    num_train_epochs=1,

    # ✅ train for only 200 optimizer steps
    max_steps=200,

    logging_steps=10,
    save_steps=50,
    save_total_limit=2,

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",
    report_to="none",

    max_length=384,

    dataset_text_field="text",
    packing=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    eval_strategy="steps",
    eval_steps=50,

    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,

    group_by_length=True
)

print(training_config)

SFTConfig(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chars_per_token=<CHARS_PER_TOKEN>,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_batch_size=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eos_token=<EOS_TOKEN>,
eval_a

In [ ]:
# ============================================================
# Cell 7 - Build SFTTrainer
# ============================================================

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer
)

print("Trainer created successfully.")

Converting train dataset to ChatML:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Trainer created successfully.


In [ ]:
# ============================================================
# Cell 8 - Train
# ============================================================

trainer.train()

Step,Training Loss,Validation Loss
50,2.052600,2.102512
100,1.993800,2.065711
150,1.951700,2.047711
200,1.979000,2.043441


TrainOutput(global_step=200, training_loss=2.083449993133545, metrics={'train_runtime': 4317.2694, 'train_samples_per_second': 0.741, 'train_steps_per_second': 0.046, 'total_flos': 2.0713965859915776e+16, 'train_loss': 2.083449993133545})

In [ ]:
# ============================================================
# Cell 9 - Save everything needed for VSCode project on Colab
# ============================================================

import os
import json
import shutil
from pathlib import Path
from google.colab import files

save_path = Path("/content/medical_phi3_lora_adapter")
zip_base_path = "/content/medical_phi3_lora_adapter"

# Remove old saved folder if exists
if save_path.exists():
    shutil.rmtree(save_path)

save_path.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter
trainer.save_model(str(save_path))

# Save tokenizer
tokenizer.save_pretrained(str(save_path))

def make_json_serializable(obj):
    if isinstance(obj, set):
        return list(obj)
    if isinstance(obj, tuple):
        return list(obj)
    if isinstance(obj, Path):
        return str(obj)
    if hasattr(obj, "item"):
        return obj.item()
    return str(obj)

metadata = {
    "base_model": "microsoft/Phi-3-mini-4k-instruct",
    "adapter_type": "LoRA / PEFT",
    "training_dataset": "ruslanmv/HealthCareMagic-100k",
    "train_size": int(TRAIN_SIZE),
    "eval_size": int(EVAL_SIZE),
    "max_steps": int(training_config.max_steps) if training_config.max_steps is not None else None,
    "max_length": int(training_config.max_length) if training_config.max_length is not None else None,
    "lora_r": int(lora_config.r),
    "lora_alpha": int(lora_config.lora_alpha),
    "lora_dropout": float(lora_config.lora_dropout),
    "target_modules": list(lora_config.target_modules) if lora_config.target_modules is not None else None,
}

with open(save_path / "training_metadata.json", "w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        indent=4,
        ensure_ascii=False,
        default=make_json_serializable
    )

readme_text = """# Medical Phi-3 LoRA Adapter

This folder contains the LoRA adapter fine-tuned from:

Base model:
microsoft/Phi-3-mini-4k-instruct

Training dataset:
ruslanmv/HealthCareMagic-100k

Use this adapter with Hugging Face Transformers + PEFT.
Do not load it alone as a full model. Load the base model first, then attach this adapter.
"""

with open(save_path / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("Saved files:")
for file in sorted(save_path.iterdir()):
    print("-", file.name)

# Create zip file
zip_file = shutil.make_archive(zip_base_path, "zip", save_path)

print("\nZIP created at:")
print(zip_file)

# Download directly to your PC
files.download(zip_file)

Saved files:
- README.md
- adapter_config.json
- adapter_model.safetensors
- added_tokens.json
- special_tokens_map.json
- tokenizer.json
- tokenizer.model
- tokenizer_config.json
- training_args.bin
- training_metadata.json

ZIP created at:
/content/medical_phi3_lora_adapter.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Cell 10 - Quick test inference - FIXED
# ============================================================

import torch

model.eval()

# Important fix for Phi-3 DynamicCache error
model.config.use_cache = False
model.generation_config.use_cache = False

prompt = """### Instruction:
You are a helpful medical assistant. Answer carefully and recommend seeing a doctor when appropriate.

### Input:
I have had a sore throat and fever for two days. What should I do?

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,

        # ✅ Fix DynamicCache get_max_length error
        use_cache=False
    )

full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "### Response:" in full_text:
    print(full_text.split("### Response:")[-1].strip())
else:
    print(full_text)

Hi, Welcome to Chat Doctor  You need not be worried as you can take paracetamol or ibuprofen in the dosage given below along with sufficient rest which will help relieve your symptoms within few hours -
